In [4]:
import os
import sys

print("Python Version:", sys.version)
print("Current Directory:", os.getcwd())

if os.path.exists("/kaggle/working"):
    print("Kaggle Environment: Ready")
else:
    print("Kaggle Environment: Not Detected")

Python Version: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Current Directory: /kaggle/working
Kaggle Environment: Ready


In [7]:
!pip install -q pypdf sentence-transformers transformers sentencepiece

In [8]:
import pypdf
import sentence_transformers
import transformers
import sentencepiece

print("All libraries installed successfully!")
print("PyPDF:", pypdf.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("Transformers:", transformers.__version__)

All libraries installed successfully!
PyPDF: 7.1.0
Sentence Transformers: 5.7.0
Transformers: 5.20.2


In [10]:
import os

print("Current directory:", os.getcwd())
print("\nFiles in working directory:")

print(os.listdir("/kaggle/working"))

Current directory: /kaggle/working

Files in working directory:
['.virtual_documents']


In [12]:
!pip install -q reportlab

from reportlab.pdfgen import canvas

pdf_path = "/kaggle/working/Week3_Sample_Study_Notes.pdf"

c = canvas.Canvas(pdf_path)

notes = [
    ("Page 1: Machine Learning",
     "Machine learning is a field of artificial intelligence. "
     "It allows computers to learn patterns from data and make predictions. "
     "The three main types are supervised, unsupervised and reinforcement learning."),

    ("Page 2: Data Preparation",
     "Data cleaning removes missing values, duplicate records and incorrect data. "
     "Feature engineering creates useful input variables. "
     "Training data teaches a model, while testing data evaluates it on unseen examples."),

    ("Page 3: Model Evaluation",
     "Overfitting occurs when a model learns training data too closely. "
     "Underfitting occurs when a model is too simple to learn important patterns. "
     "Accuracy, precision, recall and F1-score are evaluation metrics."),

    ("Page 4: Neural Networks",
     "Neural networks contain input, hidden and output layers. "
     "Activation functions include ReLU, sigmoid and tanh. "
     "Backpropagation calculates gradients used to update model parameters."),

    ("Page 5: Embeddings and RAG",
     "Text embeddings are numerical representations of text. "
     "Semantic search finds information based on meaning. "
     "Chunking divides documents into smaller sections. "
     "Retrieval-Augmented Generation retrieves relevant passages and uses them "
     "as context for generating answers. A RAG system should provide source "
     "references and avoid answering when evidence is insufficient.")
]

for title, paragraph in notes:
    c.setFont("Helvetica-Bold", 18)
    c.drawString(50, 780, title)

    c.setFont("Helvetica", 12)

    y = 730
    words = paragraph.split()
    line = ""

    for word in words:
        if len(line + " " + word) > 85:
            c.drawString(50, y, line)
            y -= 25
            line = word
        else:
            line = (line + " " + word).strip()

    if line:
        c.drawString(50, y, line)

    c.showPage()

c.save()

print("PDF created successfully!")
print("Location:", pdf_path)
print("Exists:", os.path.exists(pdf_path))

PDF created successfully!
Location: /kaggle/working/Week3_Sample_Study_Notes.pdf
Exists: True


In [13]:
from pypdf import PdfReader

# PDF location
pdf_path = "/kaggle/working/Week3_Sample_Study_Notes.pdf"

# Read PDF
reader = PdfReader(pdf_path)

# Store page text with page numbers
pages = []

for page_number, page in enumerate(reader.pages, start=1):

    text = page.extract_text() or ""

    pages.append({
        "page": page_number,
        "text": text
    })

# Display results
print("Total PDF Pages:", len(pages))

for item in pages:
    print("\n" + "=" * 50)
    print("PAGE:", item["page"])
    print(item["text"])

Total PDF Pages: 5

PAGE: 1
Page 1: Machine Learning
Machine learning is a field of artificial intelligence. It allows computers to learn
patterns from data and make predictions. The three main types are supervised,
unsupervised and reinforcement learning.


PAGE: 2
Page 2: Data Preparation
Data cleaning removes missing values, duplicate records and incorrect data. Feature
engineering creates useful input variables. Training data teaches a model, while
testing data evaluates it on unseen examples.


PAGE: 3
Page 3: Model Evaluation
Overfitting occurs when a model learns training data too closely. Underfitting occurs
when a model is too simple to learn important patterns. Accuracy, precision, recall
and F1-score are evaluation metrics.


PAGE: 4
Page 4: Neural Networks
Neural networks contain input, hidden and output layers. Activation functions include
ReLU, sigmoid and tanh. Backpropagation calculates gradients used to update model
parameters.


PAGE: 5
Page 5: Embeddings and RAG
Text

In [14]:
total_characters = sum(
    len(item["text"]) for item in pages
)

print("Total Pages:", len(pages))
print("Total Extracted Characters:", total_characters)

if len(pages) == 5 and total_characters > 0:
    print("PDF Text Extraction Successful!")
else:
    print("Please check PDF extraction.")

Total Pages: 5
Total Extracted Characters: 1290
PDF Text Extraction Successful!


In [15]:
def create_chunks(pages, chunk_size=300, overlap=50):

    chunks = []

    for item in pages:

        text = item["text"]

        start = 0

        while start < len(text):

            end = start + chunk_size

            chunk_text = text[start:end]

            chunks.append({
                "page": item["page"],
                "text": chunk_text
            })

            start += chunk_size - overlap

    return chunks


print("Chunking function created successfully!")

Chunking function created successfully!


In [16]:
chunks_300 = create_chunks(
    pages,
    chunk_size=300,
    overlap=50
)

print("Total Chunks:", len(chunks_300))

for i, chunk in enumerate(chunks_300, 1):

    print("\nChunk:", i)
    print("Page:", chunk["page"])
    print("Text:", chunk["text"])

Total Chunks: 6

Chunk: 1
Page: 1
Text: Page 1: Machine Learning
Machine learning is a field of artificial intelligence. It allows computers to learn
patterns from data and make predictions. The three main types are supervised,
unsupervised and reinforcement learning.


Chunk: 2
Page: 2
Text: Page 2: Data Preparation
Data cleaning removes missing values, duplicate records and incorrect data. Feature
engineering creates useful input variables. Training data teaches a model, while
testing data evaluates it on unseen examples.


Chunk: 3
Page: 3
Text: Page 3: Model Evaluation
Overfitting occurs when a model learns training data too closely. Underfitting occurs
when a model is too simple to learn important patterns. Accuracy, precision, recall
and F1-score are evaluation metrics.


Chunk: 4
Page: 4
Text: Page 4: Neural Networks
Neural networks contain input, hidden and output layers. Activation functions include
ReLU, sigmoid and tanh. Backpropagation calculates gradients used to update mo

In [17]:
print("Original Pages:", len(pages))
print("Generated Chunks:", len(chunks_300))

print("Chunking Successful!")

Original Pages: 5
Generated Chunks: 6
Chunking Successful!


In [18]:
from sentence_transformers import SentenceTransformer

# Load pretrained embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [19]:
# Extract text from chunks
chunk_texts = [
    chunk["text"] for chunk in chunks_300
]

# Generate numerical embeddings
embeddings = model.encode(
    chunk_texts,
    convert_to_tensor=True
)

print("Total Chunks:", len(chunk_texts))
print("Embedding Shape:", embeddings.shape)
print("Embeddings generated successfully!")

Total Chunks: 6
Embedding Shape: torch.Size([6, 384])
Embeddings generated successfully!


In [20]:
from sentence_transformers import util

# Compare the embeddings of all chunks
similarity_matrix = util.cos_sim(
    embeddings,
    embeddings
)

print("Similarity Matrix:")
print(similarity_matrix)

print("\nMatrix Shape:", similarity_matrix.shape)

Similarity Matrix:
tensor([[ 1.0000,  0.2768,  0.2715,  0.3432,  0.1637,  0.0589],
        [ 0.2768,  1.0000,  0.3817,  0.1326,  0.1547,  0.1416],
        [ 0.2715,  0.3817,  1.0000,  0.1560,  0.0873,  0.0587],
        [ 0.3432,  0.1326,  0.1560,  1.0000,  0.0971, -0.0156],
        [ 0.1637,  0.1547,  0.0873,  0.0971,  1.0000,  0.4949],
        [ 0.0589,  0.1416,  0.0587, -0.0156,  0.4949,  1.0000]])

Matrix Shape: torch.Size([6, 6])


In [22]:
def semantic_search(question, top_k=3):

    # Convert question into an embedding
    question_embedding = model.encode(
        question,
        convert_to_tensor=True
    )

    # Calculate similarity with document chunks
    scores = util.cos_sim(
        question_embedding,
        embeddings
    )[0]

    # Get the most similar chunks
    top_results = scores.topk(
        k=min(top_k, len(chunks_300))
    )

    results = []

    for score, index in zip(
        top_results.values,
        top_results.indices
    ):

        idx = index.item()

        results.append({
            "page": chunks_300[idx]["page"],
            "text": chunks_300[idx]["text"],
            "score": score.item()
        })

    return results


print("Semantic search function created successfully!")

Semantic search function created successfully!


In [23]:
question = "What is machine learning?"

results = semantic_search(question)

print("Question:", question)

for i, result in enumerate(results, 1):

    print("\nResult:", i)
    print("Page:", result["page"])
    print("Similarity Score:", round(result["score"], 3))
    print("Retrieved Text:", result["text"])

Question: What is machine learning?

Result: 1
Page: 1
Similarity Score: 0.799
Retrieved Text: Page 1: Machine Learning
Machine learning is a field of artificial intelligence. It allows computers to learn
patterns from data and make predictions. The three main types are supervised,
unsupervised and reinforcement learning.


Result: 2
Page: 3
Similarity Score: 0.341
Retrieved Text: Page 3: Model Evaluation
Overfitting occurs when a model learns training data too closely. Underfitting occurs
when a model is too simple to learn important patterns. Accuracy, precision, recall
and F1-score are evaluation metrics.


Result: 3
Page: 2
Similarity Score: 0.291
Retrieved Text: Page 2: Data Preparation
Data cleaning removes missing values, duplicate records and incorrect data. Feature
engineering creates useful input variables. Training data teaches a model, while
testing data evaluates it on unseen examples.



In [24]:
chunks_500 = create_chunks(
    pages,
    chunk_size=500,
    overlap=100
)

print("300-character chunks:", len(chunks_300))
print("500-character chunks:", len(chunks_500))

300-character chunks: 6
500-character chunks: 5


In [25]:
chunk_texts_500 = [
    chunk["text"] for chunk in chunks_500
]

embeddings_500 = model.encode(
    chunk_texts_500,
    convert_to_tensor=True
)

print("500-character chunks:", len(chunks_500))
print("Embedding Shape:", embeddings_500.shape)

500-character chunks: 5
Embedding Shape: torch.Size([5, 384])


In [31]:
def compare_chunk_sizes(question):

    question_embedding = model.encode(
        question,
        convert_to_tensor=True
    )

    # Search using 300-character chunks
    scores_300 = util.cos_sim(
        question_embedding,
        embeddings
    )[0]

    best_300 = scores_300.argmax().item()

    # Search using 500-character chunks
    scores_500 = util.cos_sim(
        question_embedding,
        embeddings_500
    )[0]

    best_500 = scores_500.argmax().item()

    print("Question:", question)

    print("\n300-character chunks:")
    print("Page:", chunks_300[best_300]["page"])
    print("Score:", round(scores_300[best_300].item(), 3))
    print("Text:", chunks_300[best_300]["text"])

    print("\n500-character chunks:")
    print("Page:", chunks_500[best_500]["page"])
    print("Score:", round(scores_500[best_500].item(), 3))
    print("Text:", chunks_500[best_500]["text"])

In [32]:
compare_chunk_sizes("What is machine learning?")

Question: What is machine learning?

300-character chunks:
Page: 1
Score: 0.799
Text: Page 1: Machine Learning
Machine learning is a field of artificial intelligence. It allows computers to learn
patterns from data and make predictions. The three main types are supervised,
unsupervised and reinforcement learning.


500-character chunks:
Page: 1
Score: 0.799
Text: Page 1: Machine Learning
Machine learning is a field of artificial intelligence. It allows computers to learn
patterns from data and make predictions. The three main types are supervised,
unsupervised and reinforcement learning.



In [33]:
test_questions = [
    "What is machine learning?",
    "What are the three main types of machine learning?",
    "What is data cleaning?",
    "What is feature engineering?",
    "Why do we divide data into training and testing sets?",
    "What is overfitting?",
    "What is underfitting?",
    "What are common model evaluation metrics?",
    "What are text embeddings?",
    "What is Retrieval-Augmented Generation?"
]

for i, question in enumerate(test_questions, start=1):
    results = semantic_search(question, top_k=1)

    print(f"\nQuestion {i}: {question}")
    print("Retrieved Page:", results[0]["page"])
    print("Similarity Score:", round(results[0]["score"], 3))
    print("Retrieved Text:", results[0]["text"])
    print("-" * 70)


Question 1: What is machine learning?
Retrieved Page: 1
Similarity Score: 0.799
Retrieved Text: Page 1: Machine Learning
Machine learning is a field of artificial intelligence. It allows computers to learn
patterns from data and make predictions. The three main types are supervised,
unsupervised and reinforcement learning.

----------------------------------------------------------------------

Question 2: What are the three main types of machine learning?
Retrieved Page: 1
Similarity Score: 0.733
Retrieved Text: Page 1: Machine Learning
Machine learning is a field of artificial intelligence. It allows computers to learn
patterns from data and make predictions. The three main types are supervised,
unsupervised and reinforcement learning.

----------------------------------------------------------------------

Question 3: What is data cleaning?
Retrieved Page: 2
Similarity Score: 0.621
Retrieved Text: Page 2: Data Preparation
Data cleaning removes missing values, duplicate records and 

In [34]:
# Use 500-character chunks for the final RAG system

final_chunks = chunks_500
final_embeddings = embeddings_500

print("Final chunk count:", len(final_chunks))
print("Final embedding shape:", final_embeddings.shape)
print("Final retrieval system is ready!")

Final chunk count: 5
Final embedding shape: torch.Size([5, 384])
Final retrieval system is ready!


In [35]:
from sentence_transformers import util

def retrieve_passages(question, top_k=3):

    # Convert question into an embedding
    question_embedding = model.encode(
        question,
        convert_to_tensor=True
    )

    # Calculate similarity with PDF chunks
    scores = util.cos_sim(
        question_embedding,
        final_embeddings
    )[0]

    # Get the best matching chunks
    top_results = scores.topk(
        k=min(top_k, len(final_chunks))
    )

    retrieved_passages = []

    for score, index in zip(
        top_results.values,
        top_results.indices
    ):
        idx = index.item()

        retrieved_passages.append({
            "page": final_chunks[idx]["page"],
            "text": final_chunks[idx]["text"],
            "score": score.item()
        })

    return retrieved_passages


# Test the retrieval function
question = "What is overfitting?"

results = retrieve_passages(question)

for i, result in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("Page:", result["page"])
    print("Similarity:", round(result["score"], 3))
    print("Text:", result["text"])


Result 1
Page: 3
Similarity: 0.661
Text: Page 3: Model Evaluation
Overfitting occurs when a model learns training data too closely. Underfitting occurs
when a model is too simple to learn important patterns. Accuracy, precision, recall
and F1-score are evaluation metrics.


Result 2
Page: 2
Similarity: 0.255
Text: Page 2: Data Preparation
Data cleaning removes missing values, duplicate records and incorrect data. Feature
engineering creates useful input variables. Training data teaches a model, while
testing data evaluates it on unseen examples.


Result 3
Page: 1
Similarity: 0.154
Text: Page 1: Machine Learning
Machine learning is a field of artificial intelligence. It allows computers to learn
patterns from data and make predictions. The three main types are supervised,
unsupervised and reinforcement learning.



In [36]:
from transformers import pipeline

generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-small"
)

print("Answer generation model loaded successfully!")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

KeyError: 'The task text2text-generation has been removed from transformers. Use the `text-generation` pipeline with a chat model instead. See the migration guide for more details: https://github.com/huggingface/transformers/blob/main/MIGRATION_GUIDE_V5.md#pipelines'

In [37]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)

generator = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("FLAN-T5 model loaded successfully!")

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  308MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

FLAN-T5 model loaded successfully!


In [38]:
def chat_with_notes(question):

    # Step 1: Retrieve relevant passages
    results = retrieve_passages(question, top_k=3)

    # Step 2: Check whether the document contains evidence
    best_score = results[0]["score"]

    if best_score < 0.35:
        return {
            "answer": "I could not find this in the document",
            "pages": []
        }

    # Step 3: Combine retrieved passages
    context = "\n".join(
        result["text"] for result in results
    )

    # Step 4: Prepare the prompt
    prompt = f"""
    Answer the question using only the information below.
    If the answer is not present, say:
    I could not find this in the document.

    Context:
    {context}

    Question:
    {question}

    Answer:
    """

    # Step 5: Convert prompt into tokens
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    # Step 6: Generate answer
    outputs = generator.generate(
        **inputs,
        max_new_tokens=100
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    ).strip()

    # Step 7: Return answer and source pages
    pages = sorted(set(
        result["page"] for result in results
    ))

    return {
        "answer": answer,
        "pages": pages
    }


print("RAG question-answering function created!")



RAG question-answering function created!


In [39]:
question = "What is machine learning?"

response = chat_with_notes(question)

print("Question:", question)
print("\nGenerated Answer:")
print(response["answer"])

print("\nSource Pages:")
print(response["pages"])

Question: What is machine learning?

Generated Answer:
a field of artificial intelligence

Source Pages:
[1, 2, 3]


In [42]:
def chat_with_notes(question):

    results = retrieve_passages(question, top_k=2)

    best_score = results[0]["score"]

    if best_score < 0.35:
        return {
            "answer": "I could not find this in the document",
            "pages": []
        }

    context = "\n".join(
        result["text"] for result in results
    )

    prompt = f"""
    Read the following study notes carefully.

    NOTES:
    {context}

    QUESTION:
    {question}

    Write a complete and clear answer using the notes.
    Include all important information related to the question.
    Do not add information that is not in the notes.

    ANSWER:
    """

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = generator.generate(
        **inputs,
        max_new_tokens=120,
        num_beams=4,
        do_sample=False
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    ).strip()

    pages = sorted(set(
        result["page"] for result in results
    ))

    return {
        "answer": answer,
        "pages": pages
    }


print("Improved RAG function created!")

Improved RAG function created!


In [43]:
question = "What is machine learning?"

response = chat_with_notes(question)

print("Question:", question)
print("\nAnswer:", response["answer"])
print("Source Pages:", response["pages"])

Question: What is machine learning?

Answer: Machine learning is a field of artificial intelligence. It allows computers to learn patterns from data and make predictions. The three main types are supervised, unsupervised and reinforcement learning. Page 3: Model Evaluation Overfitting occurs when a model learns training data too closely. Underfitting occurs when a model is too simple to learn important patterns. Accuracy, precision, recall and F1-score are evaluation metrics.
Source Pages: [1, 3]


In [44]:
def chat_with_notes(question):

    results = retrieve_passages(question, top_k=1)

    best_score = results[0]["score"]

    if best_score < 0.35:
        return {
            "answer": "I could not find this in the document",
            "pages": []
        }

    context = results[0]["text"]

    prompt = f"""
    Use only the following notes to answer the question.

    Notes:
    {context}

    Question:
    {question}

    Give a complete answer in simple language.
    Include only information relevant to the question.
    Do not repeat unrelated headings or topics.

    Answer:
    """

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = generator.generate(
        **inputs,
        max_new_tokens=100,
        num_beams=4,
        do_sample=False
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    ).strip()

    return {
        "answer": answer,
        "pages": [results[0]["page"]]
    }


print("Final RAG function updated!")


Final RAG function updated!


In [45]:
question = "What is machine learning?"

response = chat_with_notes(question)

print("Question:", question)
print("\nAnswer:", response["answer"])
print("Source Pages:", response["pages"])

Question: What is machine learning?

Answer: Machine learning is a field of artificial intelligence. It allows computers to learn patterns from data and make predictions. The three main types are supervised, unsupervised and reinforcement learning.
Source Pages: [1]


In [46]:
test_questions = [
    "What is machine learning?",
    "What are the three main types of machine learning?",
    "What is data cleaning?",
    "What is feature engineering?",
    "Why do we divide data into training and testing sets?",
    "What is overfitting?",
    "What is underfitting?",
    "What are common model evaluation metrics?",
    "What are text embeddings?",
    "What is Retrieval-Augmented Generation?"
]

for i, question in enumerate(test_questions, start=1):

    response = chat_with_notes(question)

    print("\nQuestion", i, ":", question)
    print("Answer:", response["answer"])
    print("Source Pages:", response["pages"])
    print("-" * 80)


Question 1 : What is machine learning?
Answer: Machine learning is a field of artificial intelligence. It allows computers to learn patterns from data and make predictions. The three main types are supervised, unsupervised and reinforcement learning.
Source Pages: [1]
--------------------------------------------------------------------------------

Question 2 : What are the three main types of machine learning?
Answer: supervised, unsupervised and reinforcement learning.
Source Pages: [1]
--------------------------------------------------------------------------------

Question 3 : What is data cleaning?
Answer: Data cleaning removes missing values, duplicate records and incorrect data.
Source Pages: [2]
--------------------------------------------------------------------------------

Question 4 : What is feature engineering?
Answer: Data Preparation Data cleaning removes missing values, duplicate records and incorrect data.
Source Pages: [2]
------------------------------------------

In [47]:
import re

def extractive_chat(question):

    results = retrieve_passages(question, top_k=1)
    passage = results[0]["text"]
    page = results[0]["page"]

    # Split passage into sentences
    sentences = re.split(r'(?<=[.!?])\s+', passage)

    # Convert question and sentences into embeddings
    question_embedding = model.encode(
        question,
        convert_to_tensor=True
    )

    sentence_embeddings = model.encode(
        sentences,
        convert_to_tensor=True
    )

    # Find the sentence most similar to the question
    scores = util.cos_sim(
        question_embedding,
        sentence_embeddings
    )[0]

    best_index = scores.argmax().item()
    best_score = scores[best_index].item()

    # Initial evidence check
    if best_score < 0.30:
        return {
            "answer": "I could not find this in the document",
            "pages": []
        }

    return {
        "answer": sentences[best_index].strip(),
        "pages": [page]
    }


print("Extractive answer system created!")

Extractive answer system created!


In [48]:
test_questions = [
    "What is feature engineering?",
    "Why do we divide data into training and testing sets?",
    "What is overfitting?",
    "What is underfitting?",
    "What is Retrieval-Augmented Generation?"
]

for question in test_questions:

    response = extractive_chat(question)

    print("\nQuestion:", question)
    print("Answer:", response["answer"])
    print("Source Pages:", response["pages"])
    print("-" * 70)


Question: What is feature engineering?
Answer: Feature
engineering creates useful input variables.
Source Pages: [2]
----------------------------------------------------------------------

Question: Why do we divide data into training and testing sets?
Answer: Training data teaches a model, while
testing data evaluates it on unseen examples.
Source Pages: [2]
----------------------------------------------------------------------

Question: What is overfitting?
Answer: Page 3: Model Evaluation
Overfitting occurs when a model learns training data too closely.
Source Pages: [3]
----------------------------------------------------------------------

Question: What is underfitting?
Answer: Underfitting occurs
when a model is too simple to learn important patterns.
Source Pages: [3]
----------------------------------------------------------------------

Question: What is Retrieval-Augmented Generation?
Answer: Retrieval-Augmented Generation retrieves relevant passages and uses them as conte

In [49]:
question = "Who invented the telephone?"

response = extractive_chat(question)

print("Question:", question)
print("Answer:", response["answer"])
print("Source Pages:", response["pages"])

Question: Who invented the telephone?
Answer: I could not find this in the document
Source Pages: []


In [50]:
import pandas as pd

test_questions = [
    "What is machine learning?",
    "What are the three main types of machine learning?",
    "What is data cleaning?",
    "What is feature engineering?",
    "Why do we divide data into training and testing sets?",
    "What is overfitting?",
    "What is underfitting?",
    "What are common model evaluation metrics?",
    "What are text embeddings?",
    "What is Retrieval-Augmented Generation?"
]

evaluation_results = []

for question in test_questions:

    response = extractive_chat(question)

    evaluation_results.append({
        "Question": question,
        "Retrieved Answer": response["answer"],
        "Source Pages": response["pages"],
        "Answer Found": (
            response["answer"] !=
            "I could not find this in the document"
        )
    })

results_df = pd.DataFrame(evaluation_results)

display(results_df)

print("Total Questions:", len(results_df))
print("Answers Found:", results_df["Answer Found"].sum())
print("Fallback Test: Passed")

,Question,Retrieved Answer,Source Pages,Answer Found
0,What is machine learning?,Page 1: Machine Learning\nMachine learning is ...,[1],True
1,What are the three main types of machine learn...,"The three main types are supervised,\nunsuperv...",[1],True
2,What is data cleaning?,Page 2: Data Preparation\nData cleaning remove...,[2],True
3,What is feature engineering?,Feature\nengineering creates useful input vari...,[2],True
4,Why do we divide data into training and testin...,"Training data teaches a model, while\ntesting ...",[2],True
5,What is overfitting?,Page 3: Model Evaluation\nOverfitting occurs w...,[3],True
6,What is underfitting?,Underfitting occurs\nwhen a model is too simpl...,[3],True
7,What are common model evaluation metrics?,"Accuracy, precision, recall\nand F1-score are ...",[3],True
8,What are text embeddings?,Page 5: Embeddings and RAG\nText embeddings ar...,[5],True
9,What is Retrieval-Augmented Generation?,Retrieval-Augmented Generation retrieves relev...,[5],True


Total Questions: 10
Answers Found: 10
Fallback Test: Passed


In [51]:
results_df.to_csv(
    "/kaggle/working/Week3_Evaluation_Results.csv",
    index=False
)

print("Evaluation results saved successfully!")

Evaluation results saved successfully!


In [52]:
comparison_questions = [
    "What is machine learning?",
    "What is data cleaning?",
    "What is overfitting?",
    "What are text embeddings?",
    "What is Retrieval-Augmented Generation?"
]

comparison_results = []

for question in comparison_questions:

    question_embedding = model.encode(
        question,
        convert_to_tensor=True
    )

    # Find best result for 300-character chunks
    scores_300 = util.cos_sim(
        question_embedding, embeddings
    )[0]

    best_300 = scores_300.argmax().item()

    # Find best result for 500-character chunks
    scores_500 = util.cos_sim(
        question_embedding, embeddings_500
    )[0]

    best_500 = scores_500.argmax().item()

    comparison_results.append({
        "Question": question,
        "300 Chunk Page": chunks_300[best_300]["page"],
        "300 Score": round(scores_300[best_300].item(), 3),
        "500 Chunk Page": chunks_500[best_500]["page"],
        "500 Score": round(scores_500[best_500].item(), 3)
    })

comparison_df = pd.DataFrame(comparison_results)

display(comparison_df)

,Question,300 Chunk Page,300 Score,500 Chunk Page,500 Score
0,What is machine learning?,1,0.799,1,0.799
1,What is data cleaning?,2,0.621,2,0.621
2,What is overfitting?,3,0.661,3,0.661
3,What are text embeddings?,5,0.623,5,0.612
4,What is Retrieval-Augmented Generation?,5,0.469,5,0.435


In [53]:
comparison_df.to_csv(
    "/kaggle/working/Week3_Chunk_Comparison.csv",
    index=False
)

print("Chunk comparison saved successfully!")

Chunk comparison saved successfully!
